In [ ]:
# =========================================================
# 03_validacao_estatistica.ipynb - VALIDAÇÃO ESTATÍSTICA
# =========================================================
# Objetivo:
# - Comparar INMET x ERA5
# - Calcular métricas: EAM, RMSE, ERM, R² e r
# - Gerar planilha final de validação
# =========================================================

# =========================================================
# 0) Importar bibliotecas
# =========================================================
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
from google.colab import files

# =========================================================
# 1) Ler dados INMET + ERA5 e resumo das estações
# =========================================================
df_final = pd.read_excel("07_INMET_ERA5_MENSAL_ACUM.xlsx")
resumo_estudo = pd.read_excel("/content/06_RESUMO_ESTACOES_ESTUDO.xlsx")
resumo_estudo.columns = resumo_estudo.columns.str.strip().str.replace(" ","_")

# =========================================================
# 2) Lista para armazenar resultados
# =========================================================
estatistica = []

# =========================================================
# 3) Loop por estação
# =========================================================
for nome in df_final["Nome"].unique():
    df_est = df_final[df_final["Nome"]==nome].dropna(subset=["Prec_INMET","Prec_ERA5"])
    
    if df_est.empty:
        estatistica.append([nome] + [np.nan]*8)
        continue

    # Valores INMET (x) e ERA5 (y)
    x = df_est["Prec_INMET"].values.reshape(-1,1)
    y = df_est["Prec_ERA5"].values.reshape(-1,1)

    # Limpeza de NaN e Inf
    mask = (~np.isnan(x.flatten())) & (~np.isnan(y.flatten())) & (~np.isinf(x.flatten())) & (~np.isinf(y.flatten()))
    xv, yv = x[mask].reshape(-1,1), y[mask].reshape(-1,1)

    if len(xv) == 0:
        estatistica.append([nome] + [np.nan]*8)
        continue

    # ======================
    # 4) Cálculo de métricas
    # ======================
    EAM = np.mean(np.abs(yv - xv))  # Erro absoluto médio
    RMSE = np.sqrt(mean_squared_error(xv, yv))  # Root Mean Squared Error

    # ERM (Erro relativo médio %)
    mask_nonzero = xv.flatten() != 0
    ERM = np.mean(np.abs((yv.flatten()[mask_nonzero] - xv.flatten()[mask_nonzero]) / xv.flatten()[mask_nonzero])) * 100

    # Regressão linear e R²
    lr = LinearRegression().fit(xv, yv)
    y_pred = lr.predict(xv)
    R2 = r2_score(yv, y_pred)

    # Correlação de Pearson
    r, _ = pearsonr(xv.flatten(), yv.flatten())

    # Lat/Long da estação
    lat, long = df_est["Lat"].iloc[0], df_est["Long"].iloc[0]

    # Porcentagem concluída da estação
    perc = resumo_estudo.loc[resumo_estudo["Nome"]==nome, "Porcentagem_Concluida"].values[0]

    # Armazenar resultados arredondados
    estatistica.append([nome, lat, long, perc, round(EAM,2), round(ERM,2), round(RMSE,2), round(R2,2), round(r,2)])

# =========================================================
# 5) Criar DataFrame final de validação
# =========================================================
df_estat = pd.DataFrame(estatistica, columns=[
    "Nome","Lat","Long","Porcentagem_Concluida","EAM","ERM","RMSE","R²","r"
])

# =========================================================
# 6) Exportar planilha final de validação
# =========================================================
df_estat.to_excel("08_VALIDACAO_ESTATISTICA_FINAL.xlsx", index=False)
files.download("08_VALIDACAO_ESTATISTICA_FINAL.xlsx")
print("✅ Planilha de validação final exportada com R², r, EAM, RMSE e ERM")
